In [1]:
!pip install -q -U bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 43.4 MB/s eta 0:00:00:00:0100:01


In [2]:
!pip install -q -U bitsandbytes transformers peft accelerate trl datasets rouge_score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 71.8 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 40.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 992.6/992.6 kB 51.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 82.1 MB/s eta 0:00:00:00:01


In [3]:
!pip install -U bitsandbytes accelerate


In [4]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import json
import time
import csv
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from rouge_score import rouge_scorer

HF_REPO = "l3mon3/Vietnamese_legal_dataset"
BASE_MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
ADAPTER_REPO = "/kaggle/input/datasets/tinphan2007/qwen-3b/qwen_legal_lora_rag_ft"
JSON_OUTPUT_PATH = "/kaggle/working/evaluation_results.json"
CSV_OUTPUT_PATH = "/kaggle/working/evaluate.csv"
CHECKPOINT_EVERY = 10

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_REPO, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)
model = PeftModel.from_pretrained(base_model, ADAPTER_REPO)
model.eval()
print("Model + adapter đã load xong.")

test_raw = load_dataset(HF_REPO, data_files="test.parquet", split="train")
total_test = len(test_raw)
print(f"Tổng số mẫu test: {total_test}")

PROMPT_TEMPLATE = """Bạn là một chuyên gia tư vấn pháp luật. Dựa trên các quy định của pháp luật Việt Nam được cung cấp dưới đây, hãy giải đáp câu hỏi của người dùng và BẮT BUỘC trình bày theo đúng định dạng chuẩn 3 phần sau:

**Căn cứ pháp lý:** [Tên điều luật] - [Tên văn bản luật]

**Nội dung quy định:**
[Trích dẫn chính xác nội dung điều luật áp dụng]

**Phân tích & Hướng dẫn:**
[Phân tích, áp dụng quy định trên vào tình huống của người dùng để trả lời câu hỏi]

---
CĂN CỨ PHÁP LUẬT THAM KHẢO:
{context_str}

CÂU HỎI:
{query}

TRẢ LỜI:"""

def build_context(sample):
    context_blocks = []
    for i, doc in enumerate(sample.get("retrieved_docs", [])[:2], start=1):
        context_blocks.append(f"--- [Tài liệu {i}] ---\n{doc.get('document', '')}")
    return "\n\n".join(context_blocks)

def build_synthetic_ground_truth(sample):
    law_id = sample.get("ground_truth_law_id", "")
    law_content = sample.get("ground_truth_law_content", "")
    query = sample.get("query", "")
    return f"""**Căn cứ pháp lý:** {law_id}

**Nội dung quy định:**
{law_content}

**Phân tích & Hướng dẫn:**
Căn cứ vào quy định nêu trên, đối với thắc mắc "{query}", vấn đề này được điều chỉnh và áp dụng trực tiếp theo các nguyên tắc, phạm vi của {law_id}."""

scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
output_details = []

if os.path.exists(JSON_OUTPUT_PATH):
    with open(JSON_OUTPUT_PATH, "r", encoding="utf-8") as f:
        prev = json.load(f)
    output_details = prev.get("detailed_results", [])
    print(f"Resume: đã có {len(output_details)} mẫu.")

processed_ids = {x["id"] for x in output_details}
start_time = time.time()

for idx in range(total_test):
    sample = test_raw[idx]
    sample_id = sample.get("id", str(idx))
    if sample_id in processed_ids:
        continue

    query = sample.get("query", "")
    law_id = sample.get("ground_truth_law_id", "")
    law_content = sample.get("ground_truth_law_content", "")
    context_str = build_context(sample)
    ground_truth = build_synthetic_ground_truth(sample)

    user_prompt = PROMPT_TEMPLATE.format(context_str=context_str, query=query)
    messages = [{"role": "user", "content": user_prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    model_inputs = tokenizer([text], return_tensors="pt", truncation=True, max_length=2048).to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=512,
            temperature=0.1,
            top_p=0.9,
            do_sample=False,
        )
    generated_ids = [out[len(inp):] for inp, out in zip(model_inputs.input_ids, generated_ids)]
    predicted_text = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

    is_citation_correct = False
    if "**Căn cứ pháp lý:**" in predicted_text:
        legal_basis_part = predicted_text.split("**Căn cứ pháp lý:**")[1]
        if "**Nội dung quy định:**" in legal_basis_part:
            legal_basis_part = legal_basis_part.split("**Nội dung quy định:**")[0]
        else:
            legal_basis_part = legal_basis_part.strip().split("\n")[0]
        is_citation_correct = law_id.lower() in legal_basis_part.lower()

    has_part1 = "**Căn cứ pháp lý:**" in predicted_text
    has_part2 = "**Nội dung quy định:**" in predicted_text
    has_part3 = "**Phân tích & Hướng dẫn:**" in predicted_text
    is_format_correct = has_part1 and has_part2 and has_part3

    scores = scorer.score(ground_truth, predicted_text)
    rouge_l_score = scores['rougeL'].fmeasure

    output_details.append({
        "id": sample_id,
        "query": query,
        "ground_truth_citation": law_id,
        "evidence": law_content,
        "generated_answer": predicted_text,
        "metrics": {
            "citation_accuracy": is_citation_correct,
            "format_adherence": is_format_correct,
            "rouge_l": rouge_l_score,
        }
    })
    processed_ids.add(sample_id)
    done_count = len(output_details)

    elapsed = time.time() - start_time
    avg_per_sample = elapsed / done_count
    remaining = total_test - done_count
    eta_seconds = avg_per_sample * remaining
    print(f"{done_count}/{total_test} | đã chạy: {elapsed/60:.1f} phút | trung bình: {avg_per_sample:.2f}s/mẫu | dự kiến còn lại: {eta_seconds/60:.1f} phút")

    if done_count % CHECKPOINT_EVERY == 0:
        cc = sum(1 for x in output_details if x["metrics"]["citation_accuracy"])
        cf = sum(1 for x in output_details if x["metrics"]["format_adherence"])
        tr = sum(x["metrics"]["rouge_l"] for x in output_details)
        with open(JSON_OUTPUT_PATH, "w", encoding="utf-8") as f:
            json.dump({
                "total_samples": done_count,
                "overall_metrics": {
                    "citation_accuracy_percentage": round((cc / done_count) * 100, 2),
                    "format_adherence_percentage": round((cf / done_count) * 100, 2),
                    "avg_rouge_l_percentage": round((tr / done_count) * 100, 2),
                },
                "detailed_results": output_details,
            }, f, ensure_ascii=False, indent=2)

total_samples = len(output_details)
correct_citations = sum(1 for x in output_details if x["metrics"]["citation_accuracy"])
correct_formats = sum(1 for x in output_details if x["metrics"]["format_adherence"])
total_rouge_l = sum(x["metrics"]["rouge_l"] for x in output_details)

summary_results = {
    "total_samples": total_samples,
    "overall_metrics": {
        "citation_accuracy_percentage": round((correct_citations / total_samples) * 100, 2),
        "format_adherence_percentage": round((correct_formats / total_samples) * 100, 2),
        "avg_rouge_l_percentage": round((total_rouge_l / total_samples) * 100, 2),
    },
    "detailed_results": output_details,
}
with open(JSON_OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(summary_results, f, ensure_ascii=False, indent=2)

print(f"Citation Accuracy: {summary_results['overall_metrics']['citation_accuracy_percentage']}%")
print(f"Format Adherence: {summary_results['overall_metrics']['format_adherence_percentage']}%")
print(f"ROUGE-L: {summary_results['overall_metrics']['avg_rouge_l_percentage']}%")

with open(CSV_OUTPUT_PATH, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow([
        "id", "query", "ground_truth_citation", "evidence",
        "citation_accurate", "format_adherence", "rouge_l",
        "generated_answer", "extracted_claim", "fever_label", "fever_reason"
    ])
    for item in output_details:
        writer.writerow([
            item["id"],
            item["query"],
            item["ground_truth_citation"],
            item["evidence"],
            item["metrics"]["citation_accuracy"],
            1.0 if item["metrics"]["format_adherence"] else 0.0,
            item["metrics"]["rouge_l"],
            item["generated_answer"],
            "",
            "",
            "",
        ])

print(f"Đã lưu: {CSV_OUTPUT_PATH}")

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

...

test.parquet:   0%|          | 0.00/659k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Tổng số mẫu test: 484
1/484 | đã chạy: 1.0 phút | trung bình: 57.57s/mẫu | dự kiến còn lại: 463.5 phút
2/484 | đã chạy: 2.3 phút | trung bình: 68.25s/mẫu | dự kiến còn lại: 548.3 phút
3/484 | đã chạy: 3.3 phút | trung bình: 66.72s/mẫu | dự kiến còn lại: 534.8 phút
4/484 | đã chạy: 4.5 phút | trung bình: 67.31s/mẫu | dự kiến còn lại: 538.5 phút


KeyboardInterrupt: 